# D5.5 · Proposing the policy change

**Function D — The Agentic SOC → The Agentic SOC — Recover and Root Cause**

Builds on **[D5.4 · Post-incident change surface](https://spbreed.github.io/cyber-commons/lessons/D5.4.html)**.

| | |
|---|---|
| Tools used | OPA |

## What this lesson is

**What it covers.** Turning a root cause record into a policy diff, with the incident's measured numbers as the reason, and naming what the diff does not fix.

**Why a security engineer needs it.** Most incidents change what is deployed and leave what is *allowed* untouched, so the next system built under the same policy reproduces the conditions. A diff can be argued with; a postmortem paragraph cannot. And an indicator that no policy change can fix is an engineering item — saying so in the proposal is what stops it falling between two functions.

## 1 · The hook

The incident produced a fix to what is deployed and no change at all to what is allowed. So the next system built under the same policy reproduces the same conditions, and the postmortem sits in a folder being correct.

> **At CyberTravels.** The diff is against CyberTravels' policy, and the expensive line is the one that says a tool call without provenance is refused rather than recorded. That will break CyberTravels' vendor integrations, which is precisely the review that should happen before it ships.

## 2 · The framework

```
   root cause record
        |
        v
   +--------------------------------------------------------------+
   | clause                          from -> to            why     |
   | mcp.vendor.tool_descriptions    trusted -> pinned     the     |
   | tool.call.provenance            when present ->       altered |
   |                                 required, else refuse desc    |
   | payments.refund.approval        >500 -> every amount  KCI-03  |
   +--------------------------------------------------------------+
        |
        v
   NOT addressed: KCI-04
   the policy already required 15 minutes. the control was never built
   to meet it -> engineering item, named here so it is not lost between
   the two functions
```

Most incidents end with a change to what is **deployed**. The policy that
permitted the incident is usually untouched — so the next system built under it
reproduces the conditions.

Stated as a diff, with the clause as it stands, the new text, and the incident
as the reason, the change can be argued with. Stated as a postmortem paragraph,
it cannot.

## 3 · Name what the diff does not fix

The proposal here changes three clauses and explicitly fails to address one
indicator — because no policy change can. The policy already required detection
within fifteen minutes; the control was simply never built to meet it.

That is an engineering item, not a policy item, and saying so in the proposal is
what stops it falling between the two functions forever.

## 4 · A root cause record becomes a diff

Every 'why' column is a measured number from the incident. A proposal argued on opinions gets decided by whoever is loudest.

### The skill — [`skills/grc/policy-change-proposal/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/grc/policy-change-proposal/SKILL.md)

```yaml
name: policy-change-proposal
description: >-
  Turn a root cause record into a reviewable policy diff with the incident
  attached as evidence, and name what the diff does not fix. Use at the end of an
  incident, when a postmortem's lessons never reach the policy, or when a
  proposed control change needs to be argued on evidence rather than opinion.
allowed-tools: Read, Grep, Glob
```

# The last step of an incident is a change to what is allowed

Most incidents end with a change to what is *deployed*. The policy that
permitted the incident is usually untouched, so the next system built under it
reproduces the conditions.

Stated as a diff — clause, old value, new value, and the incident as the reason
— the change can be argued with. Stated as a postmortem paragraph, it cannot.

## When to use this

After `root-cause-record` and `kci-fix-validation`, with both in hand. Before
the incident is closed, because the appetite for a policy change decays fast.

## Step-by-step

**1 — Quote the clause as it stands.** A proposal that paraphrases the current
policy is arguing with something nobody wrote.

**2 — Write the new value as text that could be adopted.** Not a direction of
travel — the words.

**3 — Put the measured number in the "why".** "41% of calls carried provenance
during the incident" ends an argument that "we should tighten provenance"
starts.

**4 — Name what the diff does NOT address.** A KCI that no policy change fixes
is an engineering item, and saying so stops it falling between the two.

**5 — Expect the expensive clause to be refused, and say so.** A proposal whose
every line is easy did not come from a real incident.

## Example

**Input** — a root cause record and four policy clauses, in
[`scripts/policy_change_proposal.py`](scripts/policy_change_proposal.py).

**Output** — one change from a real run:

```
  tool.call.provenance
  - recorded when present
  + required; a call without it is refused
    why: 41% of calls carried provenance during the incident, so it was optional
```

## Output contract

```json
{
  "incident": "str",
  "changes": [{"clause": "str", "from": "str", "to": "str", "why": "str"}],
  "unaddressed": ["str"]
}
```

## Common edge cases

- **The policy already required it.** Then the gap is engineering, not policy —
  and that is the finding.
- **The change breaks integrations.** State it in the diff. That review is the
  one that should happen before it ships.
- **No clause covers the behaviour.** A new clause is a bigger ask than an
  amendment; say which you are making.

## Failure modes

- **Proposals with an opinion in the "why" column.** They get argued on
  opinions, and the loudest person wins.
- **Silence on what is not fixed.** The unaddressed KCI is the one that recurs.
- **Deferring to "the next policy review".** By then the evidence is cold.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/grc/policy-change-proposal/scripts/policy_change_proposal.py
SCRIPT = "skills/grc/policy-change-proposal/scripts/policy_change_proposal.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; sparse-checkout then materialises only the two directories a
    # lesson needs: the procedures, and the repository they are run against.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    # `skills` is the procedures; `cybertravels` is the sample repository they
    # scan; `curriculum` and `site/data` hold the framework mapping and the
    # session list that the reference-lookup skill reads. Miss any of them and
    # the skill clones successfully and then fails on a path that is not there,
    # which is how A0.2 failed its first Kaggle run.
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set",
                    "skills", "cybertravels", "curriculum", "site/data"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

Three clauses changed with the incident as evidence, and KCI-04 named as unaddressed because it is an engineering gap rather than a policy one.

## Your turn

Write the fourth clause — the one your organisation would refuse. A proposal whose every line is easy did not come from a real incident.

---

**Next → [D5.6 · Regulatory clock](https://spbreed.github.io/cyber-commons/lessons/D5.6.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D5.5.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D5.5.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*